# ETL Pipeline and Database Benchmarks

In [ ]:
import duckdb, time, os, sys
sys.path.append('..')
import matplotlib.pyplot as plt

con = duckdb.connect('../ecommerce.duckdb')

# PERFORMANCE SETTINGS (Fix for OutOfMemory)
con.execute("PRAGMA memory_limit='8GB'")
con.execute("PRAGMA temp_directory='../duckdb_temp'")
con.execute("PRAGMA max_temp_directory_size='100GiB'")
con.execute("SET preserve_insertion_order=false")

# --- 1. Batch Size vs Throughput ---
# (Using results from our successful run to avoid re-running heavy loads)
batch_results = [
    {'batch': 10000, 'rps': 92725},
    {'batch': 50000, 'rps': 996256},
    {'batch': 100000, 'rps': 1074273},
    {'batch': 250000, 'rps': 1434894}
]

plt.figure(figsize=(10,5))
plt.plot([r['batch'] for r in batch_results], [r['rps'] for r in batch_results],
         marker='o', linewidth=2, color='steelblue')
plt.xlabel('Batch Size (rows)')
plt.ylabel('Throughput (rows/second)')
plt.title('Batch Size vs Insert Throughput')
plt.grid(True, alpha=0.3)
plt.show()

# --- 2. Query Performance ---
queries = {
    'Q1_funnel':   'SELECT p.category_main, COUNT(*) FROM fact_events e JOIN dim_product p ON e.product_key=p.product_key GROUP BY 1 LIMIT 20',
    'Q2_session':  'SELECT user_session, COUNT(*) FROM fact_events GROUP BY user_session LIMIT 10',
    'Q3_brand':    'SELECT p.brand, SUM(e.price) FROM fact_events e JOIN dim_product p ON e.product_key=p.product_key WHERE e.event_type=\'purchase\' GROUP BY 1 LIMIT 10',
    'Q5_hourly':   'SELECT d.hour, COUNT(*) FROM fact_events e JOIN dim_date d ON e.date_key=d.date_key WHERE e.event_type=\'purchase\' GROUP BY 1 ORDER BY 1'
}

times_with_idx = {}
for name, sql in queries.items():
    start = time.perf_counter()
    con.execute(sql).fetchall()
    times_with_idx[name] = round(time.perf_counter() - start, 4)
    print(f"{name}: {times_with_idx[name]}s")
